# 🎯 XGBoost Glucose Forecaster v1

**Objective:** Predict blood glucose 30 minutes into the future.  
**Model:** XGBoost (GPU) with Time-Series Split (no data leakage).  
**Evaluation:** RMSE, MAE, MAPE, Clarke Error Grid, Feature Importance.  

---

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 0: Setup
# ══════════════════════════════════════════════════════════════════════
!pip install polars xgboost plotly pyarrow scikit-learn -q

import polars as pl
import numpy as np
import xgboost as xgb
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error
from datetime import datetime, timedelta
import os
import gc
import time
import json

from google.colab import drive
drive.mount('/content/drive')

print(f"XGBoost: {xgb.__version__}")
print(f"Polars : {pl.__version__}")
print(f"Run at : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 0b: Configuration
# ══════════════════════════════════════════════════════════════════════
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"

INPUT_FILE  = f"{BASE_PATH}/processed/train_elite_physio_engineered.parquet"
MODEL_FILE  = f"{BASE_PATH}/models/xgboost_glucose_forecaster_v1.json"

# Feature definitions
FEATURE_COLS = [
    "CGM", "IOB", "COB",
    "Vel_5min", "Vel_15min", "Acc_15min",
    "Roll_Mean_60", "Roll_Std_60",
    "Hour_Sin", "Hour_Cos",
    "Insulin_Sensitivity_Risk",
]
TARGET_COL = "Target_CGM"

# Train/Test split ratio (time-series ordered)
TRAIN_RATIO = 0.80

# Ensure output directory
os.makedirs(os.path.dirname(MODEL_FILE), exist_ok=True)

print(f"📁 Input : {INPUT_FILE}")
print(f"   exists: {os.path.exists(INPUT_FILE)}")
print(f"📁 Model : {MODEL_FILE}")
print(f"\n🎯 Features: {FEATURE_COLS}")
print(f"🎯 Target  : {TARGET_COL}")

---
## STEP 1 — Data Loading & Time-Series Split (No Leakage) 📊

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 1: Load Data
# ══════════════════════════════════════════════════════════════════════
t0 = time.time()

df = pl.read_parquet(INPUT_FILE)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols in {time.time()-t0:.1f}s")
print(f"RAM   : ~{df.estimated_size('mb'):.1f} MB")
print(f"\nSchema:")
for c, dt in df.schema.items():
    print(f"  {c:30s} → {dt}")

In [ ]:
# ── Verify all feature columns exist ─────────────────────────────────
missing_cols = [c for c in FEATURE_COLS if c not in df.columns]
if missing_cols:
    print(f"⚠️  Missing columns: {missing_cols}")
    FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
    print(f"   Using: {FEATURE_COLS}")
else:
    print(f"✅ All {len(FEATURE_COLS)} feature columns present.")

# Also check for insulin-related extras that may boost performance
bonus_cols = [c for c in ["basal", "bolus", "insulin"] if c in df.columns and c not in FEATURE_COLS]
if bonus_cols:
    print(f"💡 Additional columns available (not used): {bonus_cols}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 1b: Time-Series Split
# Sort by datetime GLOBALLY (across all patients), then split 80/20
# This ensures the model is tested on "future" data
# ══════════════════════════════════════════════════════════════════════

# Sort by datetime
df = df.sort("datetime")

# Drop any remaining NaN in features/target
required = FEATURE_COLS + [TARGET_COL]
df = df.drop_nulls(subset=required)
print(f"After dropping nulls: {len(df):,} rows")

# Time-series split
split_idx = int(len(df) * TRAIN_RATIO)

df_train = df.head(split_idx)
df_test  = df.tail(len(df) - split_idx)

print(f"\n╔═══════════════════════════════════════════════════════╗")
print(f"║           📊  TIME-SERIES SPLIT                      ║")
print(f"╠═══════════════════════════════════════════════════════╣")
print(f"║  Train: {len(df_train):>12,} rows ({TRAIN_RATIO*100:.0f}%)                  ║")
print(f"║    from {str(df_train['datetime'].min())[:19]}                  ║")
print(f"║      to {str(df_train['datetime'].max())[:19]}                  ║")
print(f"║                                                       ║")
print(f"║  Test : {len(df_test):>12,} rows ({(1-TRAIN_RATIO)*100:.0f}%)                  ║")
print(f"║    from {str(df_test['datetime'].min())[:19]}                  ║")
print(f"║      to {str(df_test['datetime'].max())[:19]}                  ║")
print(f"║                                                       ║")
print(f"║  Patients in train: {df_train['id'].n_unique():>6}                           ║")
print(f"║  Patients in test:  {df_test['id'].n_unique():>6}                           ║")
print(f"╚═══════════════════════════════════════════════════════╝")

In [ ]:
# ── Convert to NumPy for XGBoost ─────────────────────────────────────
X_train = df_train.select(FEATURE_COLS).to_numpy().astype(np.float32)
y_train = df_train[TARGET_COL].to_numpy().astype(np.float32)

X_test = df_test.select(FEATURE_COLS).to_numpy().astype(np.float32)
y_test = df_test[TARGET_COL].to_numpy().astype(np.float32)

# Store test metadata for later plotting
test_ids = df_test["id"].to_list()
test_datetimes = df_test["datetime"].to_list()

# Free Polars DataFrames to save RAM
del df, df_train
gc.collect()

print(f"X_train: {X_train.shape}  ({X_train.nbytes / 1e6:.1f} MB)")
print(f"X_test:  {X_test.shape}  ({X_test.nbytes / 1e6:.1f} MB)")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

---
## STEP 2 — XGBoost GPU Training 🚀

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 2: Model Configuration & Training
# ══════════════════════════════════════════════════════════════════════

model = xgb.XGBRegressor(
    # ── Core ──────────────────────────────────────
    objective="reg:squarederror",
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=8,

    # ── GPU Acceleration ──────────────────────────
    tree_method="hist",
    device="cuda",

    # ── Regularization ────────────────────────────
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    min_child_weight=5,

    # ── Early Stopping ────────────────────────────
    early_stopping_rounds=50,
    eval_metric="rmse",

    # ── Reproducibility ───────────────────────────
    random_state=42,
    verbosity=1,
)

print("Model config:")
print(json.dumps(model.get_params(), indent=2, default=str))

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────
print("🚀 Training started...")
t_train = time.time()

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=100,  # Print every 100 rounds
)

train_time = time.time() - t_train

print(f"\n✅ Training complete in {train_time:.1f}s ({train_time/60:.1f} min)")
print(f"   Best iteration: {model.best_iteration}")
print(f"   Best RMSE:      {model.best_score:.4f}")

In [ ]:
# ── Learning Curves ──────────────────────────────────────────────────
results = model.evals_result()

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=results["validation_0"]["rmse"],
    name="Train RMSE", line=dict(color="#4ECDC4", width=1.5),
))
fig.add_trace(go.Scatter(
    y=results["validation_1"]["rmse"],
    name="Test RMSE", line=dict(color="#FF6B6B", width=1.5),
))
fig.add_vline(
    x=model.best_iteration, line_dash="dash", line_color="white",
    annotation_text=f"Best: {model.best_iteration}",
)

fig.update_layout(
    title=f"📉 Learning Curves (Stopped at {model.best_iteration} / 2000)",
    xaxis_title="Boosting Round",
    yaxis_title="RMSE (mg/dL)",
    template="plotly_dark",
    height=450,
)
fig.show()

---
## STEP 3 — Evaluation & Metrics 📈

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 3: Predictions & Metrics
# ══════════════════════════════════════════════════════════════════════
y_pred = model.predict(X_test)

# Clip predictions to clinical range
y_pred = np.clip(y_pred, 20, 500)

# ── Core Metrics ─────────────────────────────────────────────────────
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)

# MAPE (handle zeros)
mask = y_test > 0
mape = np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100

# Percentage within clinical thresholds
errors = np.abs(y_test - y_pred)
within_10 = np.mean(errors <= 10) * 100
within_15 = np.mean(errors <= 15) * 100
within_20 = np.mean(errors <= 20) * 100

print("╔══════════════════════════════════════════════════════════╗")
print("║           📊  TEST SET EVALUATION METRICS               ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  RMSE          : {rmse:>10.4f} mg/dL                    ║")
print(f"║  MAE           : {mae:>10.4f} mg/dL                    ║")
print(f"║  MAPE          : {mape:>10.4f} %                       ║")
print(f"║                                                        ║")
print(f"║  Within ±10 mg/dL : {within_10:>6.2f}%                       ║")
print(f"║  Within ±15 mg/dL : {within_15:>6.2f}%                       ║")
print(f"║  Within ±20 mg/dL : {within_20:>6.2f}%                       ║")
print(f"║                                                        ║")
print(f"║  Test Samples  : {len(y_test):>10,}                       ║")
print(f"║  Training Time : {train_time:>10.1f} seconds                ║")
print("╚══════════════════════════════════════════════════════════╝")

In [ ]:
# ── Error Distribution Histogram ─────────────────────────────────────
fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Prediction Error Distribution", "Absolute Error Distribution"))

residuals = y_pred - y_test

fig.add_trace(
    go.Histogram(x=residuals, nbinsx=150, marker_color="#4ECDC4", name="Error"),
    row=1, col=1
)
fig.add_trace(
    go.Histogram(x=errors, nbinsx=100, marker_color="#FF6B6B", name="|Error|"),
    row=1, col=2
)

fig.add_vline(x=0, line_dash="dash", line_color="white", row=1, col=1)
fig.add_vline(x=mae, line_dash="dash", line_color="yellow",
              annotation_text=f"MAE={mae:.1f}", row=1, col=2)

fig.update_layout(
    title="📊 Prediction Error Analysis",
    template="plotly_dark", height=400, showlegend=False,
)
fig.show()

---
## STEP 4 — Feature Importance 🏆

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 4: Feature Importance
# ══════════════════════════════════════════════════════════════════════
importance = model.get_booster().get_score(importance_type="gain")

# Map f0, f1, ... back to feature names
feat_imp = {}
for key, val in importance.items():
    idx = int(key.replace("f", ""))
    if idx < len(FEATURE_COLS):
        feat_imp[FEATURE_COLS[idx]] = val

# Sort
feat_imp_sorted = dict(sorted(feat_imp.items(), key=lambda x: x[1], reverse=True))

print("Feature Importance (Gain):")
print("=" * 55)
max_val = max(feat_imp_sorted.values())
for name, val in feat_imp_sorted.items():
    bar = "█" * int(val / max_val * 30)
    print(f"  {name:30s} {val:>10.1f}  {bar}")

# Plot
fig = px.bar(
    x=list(feat_imp_sorted.values()),
    y=list(feat_imp_sorted.keys()),
    orientation="h",
    title="🏆 Feature Importance (Gain)",
    labels={"x": "Gain", "y": "Feature"},
    color=list(feat_imp_sorted.values()),
    color_continuous_scale="Viridis",
)
fig.update_layout(template="plotly_dark", height=450, yaxis=dict(autorange="reversed"))
fig.show()

---
## STEP 5 — Clarke Error Grid Analysis 🩸
The clinical gold standard for evaluating glucose prediction accuracy.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 5: Clarke Error Grid
# ══════════════════════════════════════════════════════════════════════

def clarke_error_zone(ref, pred):
    """
    Classify each (reference, prediction) pair into Clarke Error Grid zones.
    
    Zones:
      A - Clinically accurate
      B - Benign errors (would not lead to wrong treatment)
      C - Overcorrection (would lead to unnecessary treatment)
      D - Failure to detect (would fail to detect hypo/hyper)
      E - Erroneous treatment (opposite treatment from needed)
    
    Parameters:
      ref  : array of reference glucose values (mg/dL)
      pred : array of predicted glucose values (mg/dL)
    
    Returns:
      zones : array of zone labels ('A', 'B', 'C', 'D', 'E')
    """
    ref = np.asarray(ref, dtype=float)
    pred = np.asarray(pred, dtype=float)
    zones = np.full(len(ref), 'D', dtype='U1')  # Default to D
    
    for i in range(len(ref)):
        r, p = ref[i], pred[i]
        
        # Zone A: within 20% or both < 70
        if (r <= 70 and p <= 70):
            zones[i] = 'A'
        elif (r >= 180 and p >= 180):
            # Both in hyperglycemic range — check 20%
            if abs(p - r) <= 0.2 * r:
                zones[i] = 'A'
            else:
                zones[i] = 'B'
        elif abs(p - r) <= 0.2 * r or (r < 70 and p < 70):
            zones[i] = 'A'
        
        # Zone E: Predicted hyper when actual hypo or vice versa
        elif (r <= 70 and p >= 180):
            zones[i] = 'E'
        elif (r >= 180 and p <= 70):
            zones[i] = 'E'
        
        # Zone C: Reference normal, predicted would cause overcorrection
        elif (r >= 70 and r <= 180 and p <= 70):
            zones[i] = 'C'
        elif (r >= 70 and r <= 180 and p >= 180):
            zones[i] = 'C'
        
        # Zone D: Reference abnormal, predicted normal (failure to detect)
        elif (r <= 70 and p > 70 and p <= 180):
            zones[i] = 'D'
        elif (r >= 180 and p > 70 and p < 180):
            zones[i] = 'D'
        
        # Everything else is Zone B
        else:
            zones[i] = 'B'
    
    return zones


def plot_clarke_error_grid(ref, pred, title="Clarke Error Grid Analysis", max_val=400, sample_n=100000):
    """
    Plot the Clarke Error Grid with zone boundaries and colored points.
    """
    # Sample for performance
    if len(ref) > sample_n:
        idx = np.random.RandomState(42).choice(len(ref), sample_n, replace=False)
        ref_s, pred_s = ref[idx], pred[idx]
    else:
        ref_s, pred_s = ref, pred

    # Calculate zones for ALL points (metrics) but plot sampled
    zones_all = clarke_error_zone(ref, pred)
    zones_sampled = clarke_error_zone(ref_s, pred_s)

    # Zone statistics (on ALL data)
    unique, counts = np.unique(zones_all, return_counts=True)
    zone_stats = dict(zip(unique, counts))
    total = len(zones_all)

    zone_colors = {'A': '#2ECC71', 'B': '#F39C12', 'C': '#E74C3C', 'D': '#9B59B6', 'E': '#E74C3C'}
    colors = [zone_colors.get(z, '#95A5A6') for z in zones_sampled]

    fig = go.Figure()

    # Plot data points
    fig.add_trace(go.Scattergl(
        x=ref_s, y=pred_s,
        mode='markers',
        marker=dict(size=2, color=colors, opacity=0.4),
        name='Predictions',
        hoverinfo='skip',
    ))

    # Perfect prediction line
    fig.add_trace(go.Scatter(
        x=[0, max_val], y=[0, max_val],
        mode='lines', line=dict(color='white', width=1, dash='dash'),
        name='Perfect', showlegend=True,
    ))

    # ── Zone boundaries ───────────────────────────────────────────────
    # Zone A upper boundary: y = 1.2x (for x > 70) and y = x + 20% of x
    # Zone A lower boundary: y = 0.8x
    x_line = np.linspace(0, max_val, 500)

    # Upper 20% line
    y_upper = np.where(x_line < 70, x_line + 15, x_line * 1.2)
    fig.add_trace(go.Scatter(x=x_line, y=y_upper,
        mode='lines', line=dict(color='rgba(46,204,113,0.5)', width=1),
        name='Zone A boundary', showlegend=False))

    # Lower 20% line
    y_lower = np.where(x_line < 70, x_line - 15, x_line * 0.8)
    fig.add_trace(go.Scatter(x=x_line, y=y_lower,
        mode='lines', line=dict(color='rgba(46,204,113,0.5)', width=1),
        name='Zone A boundary', showlegend=False))

    # Horizontal and vertical lines at 70 and 180
    for val in [70, 180]:
        fig.add_hline(y=val, line_dash='dot', line_color='rgba(255,255,255,0.2)')
        fig.add_vline(x=val, line_dash='dot', line_color='rgba(255,255,255,0.2)')

    # Zone labels
    zone_a_pct = zone_stats.get('A', 0) / total * 100
    zone_b_pct = zone_stats.get('B', 0) / total * 100
    zone_ab_pct = zone_a_pct + zone_b_pct

    fig.add_annotation(x=max_val*0.85, y=max_val*0.15, text=f"Zone A: {zone_a_pct:.1f}%",
        font=dict(size=14, color='#2ECC71'), showarrow=False)
    fig.add_annotation(x=max_val*0.85, y=max_val*0.10, text=f"Zone B: {zone_b_pct:.1f}%",
        font=dict(size=14, color='#F39C12'), showarrow=False)
    fig.add_annotation(x=max_val*0.85, y=max_val*0.05, text=f"A+B: {zone_ab_pct:.1f}%",
        font=dict(size=16, color='white'), showarrow=False)

    fig.update_layout(
        title=title,
        xaxis_title="Reference Glucose (mg/dL)",
        yaxis_title="Predicted Glucose (mg/dL)",
        template="plotly_dark",
        height=700, width=700,
        xaxis=dict(range=[0, max_val]),
        yaxis=dict(range=[0, max_val]),
    )
    fig.show()

    return zone_stats, total


print("✅ Clarke Error Grid functions defined.")

In [ ]:
# ── Generate Clarke Error Grid ───────────────────────────────────────
zone_stats, total = plot_clarke_error_grid(
    y_test, y_pred,
    title="🩸 Clarke Error Grid — XGBoost Glucose Forecaster v1 (30-min Horizon)"
)

print("\n╔══════════════════════════════════════════════════════╗")
print("║         🩸  CLARKE ERROR GRID RESULTS                ║")
print("╠══════════════════════════════════════════════════════╣")
for zone in ['A', 'B', 'C', 'D', 'E']:
    n = zone_stats.get(zone, 0)
    pct = n / total * 100
    bar = '█' * int(pct / 2)
    print(f"║  Zone {zone}: {n:>10,} ({pct:>6.2f}%)  {bar:25s}  ║")

zone_ab = (zone_stats.get('A', 0) + zone_stats.get('B', 0)) / total * 100
print(f"║                                                      ║")
status = "✅ PASS" if zone_ab > 95 else "⚠️  BELOW TARGET"
print(f"║  Zone A+B: {zone_ab:.2f}%  Target: >95%  {status:15s}   ║")
print("╚══════════════════════════════════════════════════════╝")

---
## STEP 6 — 24-Hour Prediction Trace 📉

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 6: Single Patient 24h Prediction Plot
# ══════════════════════════════════════════════════════════════════════

# Find the patient with the most data in the test set
test_id_arr = np.array(test_ids)
unique_test_ids, id_counts = np.unique(test_id_arr, return_counts=True)
best_test_patient = unique_test_ids[np.argmax(id_counts)]
print(f"Plotting patient: '{best_test_patient}' ({max(id_counts):,} test rows)")

# Get this patient's indices
mask_patient = test_id_arr == best_test_patient
patient_datetimes = np.array(test_datetimes)[mask_patient]
patient_actual = y_test[mask_patient]
patient_pred = y_pred[mask_patient]

# Pick a 24h window from the middle of the patient's data
mid = len(patient_datetimes) // 2
t_start = patient_datetimes[mid]
t_end = t_start + timedelta(hours=24)

window_mask = (patient_datetimes >= t_start) & (patient_datetimes <= t_end)
w_times = patient_datetimes[window_mask]
w_actual = patient_actual[window_mask]
w_pred = patient_pred[window_mask]

print(f"24h window: {len(w_times)} data points")
print(f"  from {w_times[0]} to {w_times[-1]}")

In [ ]:
# ── 24h Trace Plot ───────────────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=w_times, y=w_actual,
    mode='lines', name='Actual CGM',
    line=dict(color='#00D4AA', width=2),
))

fig.add_trace(go.Scatter(
    x=w_times, y=w_pred,
    mode='lines', name='Predicted (+30m)',
    line=dict(color='#FF6B6B', width=2, dash='dot'),
))

# Confidence band (±MAE)
fig.add_trace(go.Scatter(
    x=np.concatenate([w_times, w_times[::-1]]),
    y=np.concatenate([w_pred + mae, (w_pred - mae)[::-1]]),
    fill='toself',
    fillcolor='rgba(255,107,107,0.1)',
    line=dict(color='rgba(255,107,107,0)'),
    name=f'±MAE ({mae:.1f})',
))

# Clinical lines
fig.add_hline(y=70, line_dash='dash', line_color='red',
              annotation_text='Hypo (70)')
fig.add_hline(y=180, line_dash='dash', line_color='orange',
              annotation_text='Hyper (180)')

fig.update_layout(
    title=f"📉 24-Hour Prediction Trace — Patient '{best_test_patient}'",
    xaxis_title='Time',
    yaxis_title='Blood Glucose (mg/dL)',
    template='plotly_dark',
    height=550,
    legend=dict(x=0.01, y=0.99),
)
fig.show()

In [ ]:
# ── Extended 72h view ─────────────────────────────────────────────────
t_end_72 = t_start + timedelta(hours=72)
mask_72 = (patient_datetimes >= t_start) & (patient_datetimes <= t_end_72)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=patient_datetimes[mask_72], y=patient_actual[mask_72],
    mode='lines', name='Actual', line=dict(color='#00D4AA', width=1.5),
))
fig.add_trace(go.Scatter(
    x=patient_datetimes[mask_72], y=patient_pred[mask_72],
    mode='lines', name='Predicted (+30m)', line=dict(color='#FF6B6B', width=1.5, dash='dot'),
))
fig.add_hline(y=70, line_dash='dash', line_color='red')
fig.add_hline(y=180, line_dash='dash', line_color='orange')
fig.update_layout(
    title=f"📉 72-Hour View — Patient '{best_test_patient}'",
    xaxis_title='Time', yaxis_title='Glucose (mg/dL)',
    template='plotly_dark', height=450,
    xaxis_rangeslider_visible=True,
)
fig.show()

---
## STEP 7 — Save Model & Final Report 💾

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# STEP 7: Save Model
# ══════════════════════════════════════════════════════════════════════

model.save_model(MODEL_FILE)
model_size = os.path.getsize(MODEL_FILE) / (1024**2)
print(f"💾 Model saved: {MODEL_FILE}")
print(f"   File size:  {model_size:.1f} MB")

# Also save metadata
metadata = {
    "model_name": "xgboost_glucose_forecaster_v1",
    "created_at": datetime.now().isoformat(),
    "target": TARGET_COL,
    "features": FEATURE_COLS,
    "forecast_horizon_min": 30,
    "train_rows": int(X_train.shape[0]),
    "test_rows": int(X_test.shape[0]),
    "best_iteration": int(model.best_iteration),
    "metrics": {
        "rmse": round(float(rmse), 4),
        "mae": round(float(mae), 4),
        "mape": round(float(mape), 4),
        "within_10": round(float(within_10), 2),
        "within_15": round(float(within_15), 2),
        "within_20": round(float(within_20), 2),
        "clarke_zone_a_pct": round(zone_stats.get('A', 0) / total * 100, 2),
        "clarke_zone_ab_pct": round(zone_ab, 2),
    },
    "hyperparameters": {
        "n_estimators": 2000,
        "learning_rate": 0.05,
        "max_depth": 8,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "early_stopping_rounds": 50,
    },
}

meta_file = MODEL_FILE.replace(".json", "_metadata.json")
with open(meta_file, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"📋 Metadata saved: {meta_file}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FINAL REPORT
# ══════════════════════════════════════════════════════════════════════

print("")
print("╔══════════════════════════════════════════════════════════════════════╗")
print("║            🏁  XGBOOST GLUCOSE FORECASTER v1 — COMPLETE           ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print("║                                                                    ║")
print(f"║  Model File    : xgboost_glucose_forecaster_v1.json ({model_size:.1f} MB)      ║")
print(f"║  Forecast      : CGM +30 minutes                                 ║")
print(f"║  Features      : {len(FEATURE_COLS)} physiological features                       ║")
print(f"║  Best Round    : {model.best_iteration:>6} / 2000                                ║")
print(f"║  Training Time : {train_time:>8.1f}s                                        ║")
print("║                                                                    ║")
print("║  ── ACCURACY METRICS ─────────────────────────────────────────     ║")
print(f"║  RMSE          : {rmse:>8.4f} mg/dL                                  ║")
print(f"║  MAE           : {mae:>8.4f} mg/dL                                  ║")
print(f"║  MAPE          : {mape:>8.4f}%                                      ║")
print("║                                                                    ║")
print("║  ── CLINICAL ACCURACY ────────────────────────────────────────     ║")
print(f"║  Clarke Zone A : {zone_stats.get('A',0)/total*100:>8.2f}% (Clinically Accurate)           ║")
print(f"║  Clarke A+B    : {zone_ab:>8.2f}% (Target: >95%)                       ║")
print(f"║  Within ±10    : {within_10:>8.2f}%                                      ║")
print(f"║  Within ±15    : {within_15:>8.2f}%                                      ║")
print(f"║  Within ±20    : {within_20:>8.2f}%                                      ║")
print("║                                                                    ║")

# Top 3 features
top3 = list(feat_imp_sorted.keys())[:3]
print(f"║  ── TOP PREDICTORS ──────────────────────────────────────────     ║")
for i, f in enumerate(top3):
    print(f"║  #{i+1}  {f:30s}                                ║")

print("║                                                                    ║")
overall = "✅ PRODUCTION READY" if zone_ab > 95 and rmse < 25 else "⚠️  NEEDS TUNING"
print(f"║  VERDICT: {overall:55s}   ║")
print("║                                                                    ║")
print("╚══════════════════════════════════════════════════════════════════════╝")

# Cleanup
gc.collect()
print("\n🧹 Done.")